# 数据处理

下载和读取， 参考：<https://github.com/karpathy/nanoGPT/blob/master/data/shakespeare/prepare.py>

```python
import os
import requests

# os.getcwd()    会放到 code文件夹下面，而不是 zero_gpt文件夹下面
# __file__(当前文件路径)，会放到和当前文件同一个文件夹下，但是jupyter里不能用
input_file_path = os.path.join(os.path.dirname(os.getcwd()), 'input.txt')
if not os.path.exists(input_file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(input_file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(data_url).text)

# 网络不好的话还是自己手动下载放好吧
```

## 读取数据

In [4]:
text = open("input.txt", 'r').read()
print("length of datasets in characters: ", len(text))
print(text[:1000])

length of datasets in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hun

## 分词器构建

In [8]:
# 统计字典的字符数量
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
char_ascii_num = [ord(char) for char in chars]  # chr()和ord() ord() 把字符转成数字编码，chr() 把数字编码转回字符‌
print(char_ascii_num)
# 第一个字符应该是Line Feed（直译为“送纸”或“行推进”）。
# Line Feed (LF)，将光标移动到下一行的相同水平位置（在现代计算机中通常默认也会回到行首）。
# Carriage Return (CR, 回车，ASCII 13)
# 10是换行  32是space空格 33是！
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[10, 32, 33, 36, 38, 39, 44, 45, 46, 51, 58, 59, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122]
65


In [10]:
# 字符到数字的映射， 和数字到字符的映射
# 最简单的分词 tokenizer过程
# 这里和以前不一样了，以前会加一个 . 作为终止符
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]  # lambda函数， s表示sentence/string 把一串字符编码为整数列表
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hello world!"))
print(decode(encode("hello world!")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
hello world!


还有很多分词方法，比如：
+ [openai/tiktoken](https://github.com/openai/tiktoken)
+ [google/sentencepiece](https://github.com/google/sentencepiece)

In [13]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print(enc.n_vocab)

print(enc.encode("hello world!"))
print(enc.decode([31373, 995, 0]))

50257
[31373, 995, 0]
hello world!


## 构造数据集

In [15]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)  # torch.long，即 int64
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [16]:
# 划分训练集和验证集
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [17]:
# 限制最大输入长度，因为不可能一下子把 1115394 这整个文档的char全都送到网络里去
block_size = 8
train_data[: block_size+1] 
# 这里加1表示的是用前8个字符预测第9个字符，即：在构造的数据集样本中，输入是前8个字符，输出是要预测的下一个字符
# 由于滑动窗口的存在，这里看似是9个字符，实际上包含了8个样本，全为空不算，所以可以理解为 7个空+第一个字符→第二个字符，...  0个空+8个字符→最后一个字符
# 其实快速判断的方法就是看 输出有多少种样本，很明显，除了第一个字符之外，其余8个都可以作为输出，所以有8个样本

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [19]:
demo_x = train_data[: block_size]
demo_y = train_data[1: block_size+1]
for t in range(block_size):
    context = demo_x[:t+1]  # t从0开始 [:1] 相当于取0 
    target = demo_y[t]
    print(f"when input is [{context}], the target is [{target}]")

when input is [tensor([18])], the target is [47]
when input is [tensor([18, 47])], the target is [56]
when input is [tensor([18, 47, 56])], the target is [57]
when input is [tensor([18, 47, 56, 57])], the target is [58]
when input is [tensor([18, 47, 56, 57, 58])], the target is [1]
when input is [tensor([18, 47, 56, 57, 58,  1])], the target is [15]
when input is [tensor([18, 47, 56, 57, 58,  1, 15])], the target is [47]
when input is [tensor([18, 47, 56, 57, 58,  1, 15, 47])], the target is [58]


In [20]:
a = torch.randint(300, (8,))
a

tensor([ 30, 249, 228,   3, 238,  80, 114, 130])

In [22]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    """
    split：数据集划分，例如：train_data/val_data
    """
    data = train_data if split=="train" else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))  # low=0, high, size(tuple) 即：在0~1115394-8 这堆数里，生成4个索引值 这4个值是随机抽的，不是连续的，和data的连续的数字映射无关
    x = torch.stack([data[i:i+block_size] for i in ix]) # 默认dim = 0，堆叠成多行
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) 
    # y等于x向右偏移1个，所以加1就行，这里和之前的makemore不同，之前是输入3个，输出1个；现在是输入3个，输出4个(输入的3个+预测的1个)
    # 这个构建方法就和大模型/基于Transformer的预测是一致的了
    return x,y

xb,yb = get_batch('train')
print(f"inputs: {xb}\n{xb.shape}")
print(f"outputs: {yb}\n{yb.shape}")
print("-----")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is [{context}], the target is [{target}]")

inputs: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
outputs: tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
torch.Size([4, 8])
-----
when input is [tensor([24])], the target is [43]
when input is [tensor([24, 43])], the target is [58]
when input is [tensor([24, 43, 58])], the target is [5]
when input is [tensor([24, 43, 58,  5])], the target is [57]
when input is [tensor([24, 43, 58,  5, 57])], the target is [1]
when input is [tensor([24, 43, 58,  5, 57,  1])], the target is [46]
when input is [tensor([24, 43, 58,  5, 57,  1, 46])], the target is [43]
when input is [tensor([24, 43, 58,  5, 57,  1, 46, 43])], the target is [39]
when input is [tensor([44])], the target is [53]
when input is [tensor([44, 53])], the target is [5